### RAG with Vector Search 
___

Earlier, we built a RAG pipeline with three steps:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

The search step used keyword search. Now we swap in vector search. Because RAG is modular, search is the only step we touch. Build prompt and the LLM call stay exactly as they were.


### Using RAGBase class
---

we already put all the RAG logic into a **RAGBase** helper class. It has `search`, `build_prompt`, and `llm` methods, so we only need to override `search`.

In [23]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Create the OpenAI client from the environment variable
openai_client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

In [24]:
# Next, download and index the data:

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [25]:
# Then use the RAGBase class

from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [26]:
# Ask it a question:

query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

# This still uses keyword search. Text search isn't bad here, so the answer may already look right. Next we replace search with vector search.

'Yes, you can still sign up for the program. Just keep in mind that if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

len(texts)


1368

In [20]:
# First we import tqdm to see the progress
from tqdm import tqdm

# Next we chunk the texts into batches of 50 and encode each batch

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

# We end up with 1368 vectors.

100%|██████████| 28/28 [00:09<00:00,  2.87it/s]


1368

In [22]:
import numpy as np
X = np.array(vectors)

X.shape

(1368, 384)

In [29]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

We already have:

- All the indexed documents `documents`
- The embeddings matrix `X` with all these documents
- The vector search engine `vindex`


We can't pass `vindex` to RAG as-is. Text search takes the query string directly, but vector search needs the query as a vector first. So we subclass `RAGBase` and override `search` to encode the query before searching.

In [31]:
# The __init__ method adds one extra argument, embedder, for the sentence transformer. 
# Inside search we use it to turn the query into a vector. Then we query vindex with that vector instead of the raw text. 
# Everything else is inherited from RAGBase.


class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [30]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [32]:
# Try it with different queries:

vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up and join the course even if it has already begun. You can start learning and submitting homework while the submission form is open. However, if you want to receive a certificate, be sure to submit your project before submissions are no longer accepted.'